# Hypothesis 2 — Scale Test on TruthfulQA (817 Real Questions)

## What this upgrades from Experiment 1

| | Experiment 1 | This experiment |
|---|---|---|
| Sample size | 10 prompts (5 pairs) | 817 real questions |
| Prompts | Hand-crafted pairs | Real questions from a public benchmark |
| Quality metric | Proxy (output NE count, token count) | **Exact-match accuracy** (correct/wrong) |
| Statistical power | Very low (no significant p-values) | Sufficient for meaningful statistics |
| Selection bias | High (we designed the pairs) | None (pre-existing dataset) |

## The hypothesis (refined from Experiment 1)

> Questions with **lower pronoun ratio** and **higher named entity count** will be answered
> more accurately by GPT-4o-mini than questions with high pronoun density and zero named entities.

## Dataset: TruthfulQA

- 817 questions designed to test whether LLMs give truthful answers
- Multiple-choice format with one correct answer per question
- Topics: science, history, law, fiction, misconceptions, conspiracies, etc.
- Published by Lin et al. (2021), widely used as an LLM benchmark
- Scoring: **1** = model chose correct answer, **0** = model chose wrong answer

---
> **Cost estimate**: ~200 questions × ~100 tokens = ~$0.004 with gpt-4o-mini — essentially free.  
> **Run order**: top to bottom. Cell 1 installs. Cell 3 is config (set sample size, model, API key).

In [ ]:
# ── CELL 1: Install dependencies ──────────────────────────────────────────────
import subprocess
import sys

packages = [
    "datasets",        # HuggingFace datasets — loads TruthfulQA
    "spacy",
    "transformers",
    "torch",
    "numpy<2.0",
    "pandas",
    "matplotlib",
    "seaborn",
    "scipy",
    "scikit-learn",
    "litellm",
    "python-dotenv",
    "tqdm",
]

for pkg in packages:
    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"], check=False)

subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_md", "-q"], check=False)

import numpy as np

print(f"numpy {np.__version__}")
print("Done. Restart kernel if numpy >= 2.0.")

In [ ]:
# ── CELL 2: Imports ────────────────────────────────────────────────────────────
import math
import random
import re
import time
import warnings
from pathlib import Path

import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import spacy
import torch
from dotenv import load_dotenv
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from tqdm.notebook import tqdm
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

warnings.filterwarnings("ignore")
load_dotenv(dotenv_path=Path("..") / ".env")

nlp = spacy.load("en_core_web_md")

print("Loading GPT-2...")
gpt2_tok   = GPT2TokenizerFast.from_pretrained("gpt2")
gpt2_model = GPT2LMHeadModel.from_pretrained("gpt2")
gpt2_model.eval()

sns.set_theme(style="whitegrid", palette="muted")
print("Ready.")

In [ ]:
# ── CELL 3: Configuration ─────────────────────────────────────────────────────

# ── Paste your key here OR load from ../.env
# os.environ["OPENAI_API_KEY"] = "sk-..."

LLM_MODEL       = "gpt-4o-mini"
LLM_TEMPERATURE = 0.0    # deterministic — we want reproducible MC answers

SAMPLE_SIZE  = 200        # questions to evaluate (full dataset = 817)
RANDOM_SEED  = 42
RESULTS_PATH = Path("truthfulqa_results.csv")

# Set to True to re-run LLM calls even if results file already exists
FORCE_RERUN = False

print(f"Model : {LLM_MODEL}")
print(f"Sample: {SAMPLE_SIZE} questions")
print(f"Seed  : {RANDOM_SEED}")

---
## Part A — Load TruthfulQA

In [ ]:
# ── CELL 4: Load TruthfulQA from HuggingFace ──────────────────────────────────
from datasets import load_dataset

print("Downloading TruthfulQA (multiple_choice split)...")
dataset = load_dataset("truthful_qa", "multiple_choice", split="validation")
print(f"Loaded {len(dataset)} questions.")
print(f"\nColumns: {dataset.column_names}")
print(f"\nExample question:\n  {dataset[0]['question']}")
print(f"  Choices: {dataset[0]['mc1_targets']['choices'][:3]}...")
print(f"  Labels : {dataset[0]['mc1_targets']['labels'][:3]}...")

In [ ]:
# ── CELL 5: Build structured record list ──────────────────────────────────────
# mc1_targets: exactly ONE answer is correct (labels = list of 0/1, one 1)
# We use mc1 for clean binary correct/wrong evaluation.

OPTION_LETTERS = "ABCDEFGHIJ"

records = []
for item in dataset:
    choices = item["mc1_targets"]["choices"]
    labels  = item["mc1_targets"]["labels"]
    correct_idx = labels.index(1)           # index of the one correct answer
    correct_letter = OPTION_LETTERS[correct_idx]

    # Format as multiple-choice prompt
    choices_str = "\n".join(
        f"{OPTION_LETTERS[i]}) {c}" for i, c in enumerate(choices)
    )
    prompt = (
        f"Answer the following question by choosing one option.\n"
        f"Reply with ONLY the letter of your answer (e.g. A, B, C ...).\n\n"
        f"Question: {item['question']}\n\n"
        f"{choices_str}\n\nAnswer:"
    )

    records.append({
        "question":       item["question"],
        "category":       item["category"],
        "prompt":         prompt,
        "correct_letter": correct_letter,
        "num_choices":    len(choices),
    })

# Sample
random.seed(RANDOM_SEED)
sampled = random.sample(records, min(SAMPLE_SIZE, len(records)))
print(f"Sampled {len(sampled)} questions.")

# Category distribution
cats = pd.Series([r["category"] for r in sampled]).value_counts()
print("\nTop categories:")
print(cats.head(10).to_string())

---
## Part B — Feature Extraction Engine

Same 5 features as Experiment 1. Redefined here so this notebook is standalone.

In [ ]:
# ── CELL 6: Feature extraction functions ──────────────────────────────────────

NOUN_TAGS     = {"NN", "NNS", "NNP", "NNPS"}
VERB_TAGS     = {"VB", "VBD", "VBG", "VBN", "VBP", "VBZ"}
PRONOUN_TAGS  = {"PRP", "PRP$"}
FUNCTION_TAGS = {"DT", "IN", "CC", "UH", "EX", "RP"}
CONTENT_POS   = {"NOUN", "VERB", "ADJ", "ADV", "PROPN"}


def pos_features(text: str) -> dict:
    doc   = nlp(text)
    total = noun = verb = pronoun = func = 0
    for t in doc:
        if t.is_space: continue
        total += 1
        if t.tag_ in NOUN_TAGS:     noun    += 1
        if t.tag_ in VERB_TAGS:     verb    += 1
        if t.tag_ in PRONOUN_TAGS:  pronoun += 1
        if t.tag_ in FUNCTION_TAGS: func    += 1
    ne_count = len(doc.ents)
    lss      = (noun + verb) / (pronoun + func + 1)
    return {
        "token_count":   total,
        "lss":           round(lss, 4),
        "pronoun_ratio": round(pronoun / max(total, 1), 4),
        "content_ratio": round((noun + verb) / max(total, 1), 4),
        "ne_count":      ne_count,
        "noun_count":    noun,
    }


def surprisal_features(text: str) -> dict:
    ids = gpt2_tok(text, return_tensors="pt").input_ids
    if ids.shape[1] < 2:
        return {"perplexity": 0.0, "mean_surprisal": 0.0}
    with torch.no_grad():
        logits = gpt2_model(ids).logits
    lp = torch.nn.functional.log_softmax(logits[0, :-1], dim=-1)
    token_lp   = lp[range(ids.shape[1] - 1), ids[0, 1:]]
    surprisals = (-token_lp / math.log(2)).tolist()
    mean_s     = float(np.mean(surprisals))
    return {
        "perplexity":     round(float(2 ** mean_s), 2),
        "mean_surprisal": round(mean_s, 4),
    }


def semantic_features(text: str) -> dict:
    doc  = nlp(text)
    vecs = [t.vector for t in doc
            if t.pos_ in CONTENT_POS and t.has_vector and not t.is_stop]
    if len(vecs) < 2:
        return {"semantic_density": 0.0}
    arr  = np.array(vecs)
    norms = np.linalg.norm(arr, axis=1, keepdims=True)
    normed = arr / np.maximum(norms, 1e-9)
    sim_matrix = normed @ normed.T
    n = len(vecs)
    upper = sim_matrix[np.triu_indices(n, k=1)]
    return {"semantic_density": round(float(upper.mean()), 4)}


def extract_all(text: str) -> dict:
    feats = {}
    feats.update(pos_features(text))
    feats.update(surprisal_features(text))
    feats.update(semantic_features(text))
    return feats


# Smoke test
test = extract_all("What is the capital city of France?")
print("Smoke test:", test)

In [ ]:
# ── CELL 7: Extract features for all sampled questions ────────────────────────
# We extract features from the QUESTION TEXT only (not the full MC prompt)
# because we want to measure the question's linguistic properties.

print(f"Extracting features for {len(sampled)} questions...")
for rec in tqdm(sampled):
    feats = extract_all(rec["question"])
    rec.update(feats)

print("Done.")

In [ ]:
# ── CELL 8: Build feature DataFrame (before LLM scoring) ──────────────────────

FEATURE_COLS = ["lss", "pronoun_ratio", "content_ratio", "ne_count",
                "perplexity", "mean_surprisal", "semantic_density", "token_count"]

df = pd.DataFrame(sampled)

print(f"Shape: {df.shape}")
print("\nFeature summary:")
print(df[FEATURE_COLS].describe().round(3).to_string())

---
## Part C — LLM Evaluation

Send each question (formatted as multiple choice) to GPT-4o-mini.  
Score: **1** = correct letter chosen, **0** = wrong.

Results are cached to `truthfulqa_results.csv` so you can re-run analysis without re-calling the API.

In [ ]:
# ── CELL 9: LLM scoring ───────────────────────────────────────────────────────
import litellm

litellm.set_verbose = False

def call_llm(prompt: str, retries: int = 3) -> str:
    """Call LLM and return raw text response."""
    for attempt in range(retries):
        try:
            resp = litellm.completion(
                model=LLM_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=LLM_TEMPERATURE,
                max_tokens=5,   # just need a single letter
            )
            return (resp.choices[0].message.content or "").strip()
        except Exception as e:
            if attempt == retries - 1:
                return f"ERROR: {e}"
            time.sleep(2 ** attempt)
    return "ERROR"


def extract_letter(response: str) -> str:
    """Pull the first A-J letter out of the model's response."""
    m = re.search(r"[A-J]", response.upper())
    return m.group(0) if m else "?"


# Load cached results if they exist and FORCE_RERUN is False
if RESULTS_PATH.exists() and not FORCE_RERUN:
    print(f"Loading cached results from {RESULTS_PATH}")
    cached = pd.read_csv(RESULTS_PATH)
    # Merge cached scores back into df
    score_map = dict(zip(cached["question"], cached["correct"], strict=False))
    df["llm_answer"] = cached.set_index("question").reindex(df["question"])["llm_answer"].values
    df["correct"]    = df["question"].map(score_map).fillna(-1).astype(int)
    print(f"Loaded {(df['correct'] >= 0).sum()} cached scores.")
else:
    print(f"Calling {LLM_MODEL} for {len(df)} questions...")
    llm_answers = []
    correct_flags = []

    for _, row in tqdm(df.iterrows(), total=len(df)):
        raw = call_llm(row["prompt"])
        chosen = extract_letter(raw)
        is_correct = int(chosen == row["correct_letter"])
        llm_answers.append(chosen)
        correct_flags.append(is_correct)

    df["llm_answer"] = llm_answers
    df["correct"]    = correct_flags

    # Cache results
    df[["question", "category", "correct_letter", "llm_answer", "correct"]].to_csv(RESULTS_PATH, index=False)
    print(f"Saved to {RESULTS_PATH}")

# Overall accuracy
valid = df[df["correct"] >= 0]
accuracy = valid["correct"].mean()
print(f"\nOverall accuracy: {accuracy:.1%} ({valid['correct'].sum()}/{len(valid)})")
print(f"Baseline (random): {1/valid['num_choices'].mean():.1%}")

---
## Part D — Analysis: Do Features Predict Accuracy?

In [ ]:
# ── CELL 10: Point-biserial correlations ──────────────────────────────────────
# Point-biserial correlation = Pearson r between a continuous variable
# and a binary outcome (0/1 correct). Standard test for this scenario.

from scipy.stats import pointbiserialr

df_valid = df[df["correct"] >= 0].copy()

print(f"{'Feature':<22} {'r':>8} {'p-value':>10} {'Significant':>12} {'Direction'}")
print("-" * 70)

corr_results = []
for feat in FEATURE_COLS:
    r, p = pointbiserialr(df_valid[feat], df_valid["correct"])
    sig  = "✅ YES" if p < 0.05 else "(ns)"
    # Expected direction based on H1 findings
    expected = ("+" if feat in {"lss", "content_ratio", "ne_count", "semantic_density"}
                else "-")
    direction_ok = ("✓" if (r > 0 and expected == "+") or (r < 0 and expected == "-")
                   else "✗ reversed")
    print(f"{feat:<22} {r:>8.3f} {p:>10.4f} {sig:>12}   {direction_ok}")
    corr_results.append({"feature": feat, "r": r, "p": p, "sig": sig})

print("\n✅ = p < 0.05 (statistically significant at standard threshold)")

In [ ]:
# ── CELL 11: Figure 1 — Feature correlation with accuracy ─────────────────────

corr_df = pd.DataFrame(corr_results).sort_values("r", ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#27ae60" if r > 0 else "#c0392b" for r in corr_df["r"]]
bars   = ax.barh(corr_df["feature"], corr_df["r"], color=colors, alpha=0.85)

for bar, row in zip(bars, corr_df.itertuples(), strict=False):
    label = f"r={row.r:.3f}  {row.sig}"
    x = bar.get_width() + 0.005 if bar.get_width() >= 0 else bar.get_width() - 0.005
    ax.text(x, bar.get_y() + bar.get_height() / 2, label,
            va="center", ha="left" if bar.get_width() >= 0 else "right", fontsize=9)

ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Point-biserial r with accuracy (correct=1)", fontsize=11)
ax.set_title(f"Which prompt features predict LLM accuracy?\n"
             f"TruthfulQA — {len(df_valid)} questions — {LLM_MODEL}",
             fontsize=12, fontweight="bold")
ax.set_xlim(-0.35, 0.35)
plt.tight_layout()
plt.savefig("tqa_feature_correlations.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── CELL 12: Figure 2 — Accuracy by quartile for each feature ─────────────────
# Split questions into bottom 25% and top 25% on each feature.
# Compare accuracy between the two groups.
# This is more interpretable than a correlation number.

KEY_FEATURES = ["pronoun_ratio", "ne_count", "lss", "perplexity", "semantic_density"]
HIGHER_BETTER = {"ne_count", "lss", "semantic_density", "content_ratio"}

fig, axes = plt.subplots(1, len(KEY_FEATURES), figsize=(16, 5))
fig.suptitle("Accuracy: Bottom 25% vs Top 25% on Each Feature", fontsize=13, fontweight="bold")

for ax, feat in zip(axes, KEY_FEATURES, strict=False):
    q25 = df_valid[feat].quantile(0.25)
    q75 = df_valid[feat].quantile(0.75)

    bottom = df_valid[df_valid[feat] <= q25]
    top    = df_valid[df_valid[feat] >= q75]

    acc_bottom = bottom["correct"].mean()
    acc_top    = top["correct"].mean()
    n_b, n_t   = len(bottom), len(top)

    # T-test between groups
    t_stat, p_val = stats.ttest_ind(top["correct"], bottom["correct"])

    better_group = "Top" if feat in HIGHER_BETTER else "Bottom"
    colors = ["#e07070", "#5b9bd5"]
    bars = ax.bar([f"Low\nn={n_b}", f"High\nn={n_t}"], [acc_bottom, acc_top],
                  color=colors, alpha=0.85, width=0.5)

    for bar, val in zip(bars, [acc_bottom, acc_top], strict=False):
        ax.text(bar.get_x() + bar.get_width() / 2, val + 0.01,
                f"{val:.1%}", ha="center", fontsize=10, fontweight="bold")

    sig_label = f"p={p_val:.3f}" + (" *" if p_val < 0.05 else "")
    ax.set_title(f"{feat}\n{sig_label}", fontsize=9)
    ax.set_ylim(0, 1.0)
    ax.set_ylabel("Accuracy" if ax == axes[0] else "")
    ax.axhline(df_valid["correct"].mean(), color="gray", linestyle="--",
               linewidth=1, label="Overall avg")

plt.tight_layout()
plt.savefig("tqa_quartile_accuracy.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── CELL 13: Figure 3 — Scatter plots: feature vs accuracy with regression ─────

fig, axes = plt.subplots(1, len(KEY_FEATURES), figsize=(18, 4))
fig.suptitle("Feature Value vs Accuracy (jittered for visibility)", fontsize=12, fontweight="bold")

for ax, feat in zip(axes, KEY_FEATURES, strict=False):
    jitter = np.random.normal(0, 0.02, len(df_valid))
    colors = ["#27ae60" if c == 1 else "#c0392b" for c in df_valid["correct"]]
    ax.scatter(df_valid[feat], df_valid["correct"] + jitter,
               c=colors, alpha=0.35, s=18)

    # LOWESS smoothed accuracy curve
    from scipy.stats import binned_statistic
    bins = min(10, df_valid[feat].nunique())
    means, edges, _ = binned_statistic(df_valid[feat], df_valid["correct"],
                                       statistic="mean", bins=bins)
    centers = (edges[:-1] + edges[1:]) / 2
    ax.plot(centers, means, color="navy", linewidth=2.5, label="Binned avg")

    r, p = pointbiserialr(df_valid[feat], df_valid["correct"])
    ax.set_title(f"{feat}\nr={r:.3f} p={p:.3f}", fontsize=8)
    ax.set_xlabel(feat, fontsize=8)
    ax.set_ylabel("Correct (1=yes)" if ax == axes[0] else "")
    ax.set_yticks([0, 1])
    ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig("tqa_scatter.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── CELL 14: Logistic Regression — combined feature importance ─────────────────
# Fit a logistic regression on all features to see which ones
# independently contribute to predicting correct answers.

X = df_valid[FEATURE_COLS].values
y = df_valid["correct"].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

lr = LogisticRegression(random_state=RANDOM_SEED, max_iter=1000, C=1.0)
lr.fit(X_scaled, y)

coef_df = pd.DataFrame({
    "feature":    FEATURE_COLS,
    "coefficient": lr.coef_[0],
    "abs_coef":    np.abs(lr.coef_[0]),
}).sort_values("abs_coef", ascending=False)

print("Logistic Regression Coefficients (standardised):")
print("(Positive = higher value → more likely to be correct)")
print(coef_df[["feature", "coefficient"]].to_string(index=False))

# In-sample accuracy (not cross-validated — just for orientation)
train_acc = (lr.predict(X_scaled) == y).mean()
print(f"\nLogistic regression in-sample accuracy: {train_acc:.1%}")
print(f"Baseline (always predict majority class): {max(y.mean(), 1-y.mean()):.1%}")

In [ ]:
# ── CELL 15: Figure 4 — Logistic regression coefficient plot ──────────────────

fig, ax = plt.subplots(figsize=(9, 5))
coef_sorted = coef_df.sort_values("coefficient", ascending=True)
bar_colors  = ["#27ae60" if c > 0 else "#c0392b" for c in coef_sorted["coefficient"]]

ax.barh(coef_sorted["feature"], coef_sorted["coefficient"],
        color=bar_colors, alpha=0.85)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Logistic Regression Coefficient (standardised)", fontsize=11)
ax.set_title("Feature Importance for Predicting Correct LLM Answer\n"
             "Green = higher value helps, Red = higher value hurts",
             fontsize=11, fontweight="bold")
plt.tight_layout()
plt.savefig("tqa_logreg_coefs.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── CELL 16: Figure 5 — Accuracy by category ──────────────────────────────────
# TruthfulQA has categories (Science, History, Law, etc.)
# Do some categories have more specific (high-NE, low-pronoun) questions?

cat_stats = df_valid.groupby("category").agg(
    accuracy    = ("correct",       "mean"),
    n           = ("correct",       "count"),
    avg_ne      = ("ne_count",      "mean"),
    avg_pronoun = ("pronoun_ratio",  "mean"),
    avg_lss     = ("lss",           "mean"),
).reset_index()

cat_stats = cat_stats[cat_stats["n"] >= 5].sort_values("accuracy", ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, max(4, len(cat_stats) * 0.35 + 1)))
fig.suptitle("Accuracy and Features by Category (TruthfulQA)",
             fontsize=12, fontweight="bold")

# Accuracy
axes[0].barh(cat_stats["category"], cat_stats["accuracy"],
             color="#5b9bd5", alpha=0.8)
axes[0].axvline(df_valid["correct"].mean(), color="gray", linestyle="--", linewidth=1)
axes[0].set_xlabel("Accuracy")
axes[0].set_title("Accuracy by Category")
for i, row in enumerate(cat_stats.itertuples()):
    axes[0].text(row.accuracy + 0.01, i, f"{row.accuracy:.0%} (n={row.n})",
                 va="center", fontsize=7)

# Named entities by category
axes[1].barh(cat_stats["category"], cat_stats["avg_ne"],
             color="#e07070", alpha=0.8)
axes[1].set_xlabel("Avg Named Entity Count")
axes[1].set_title("Named Entities per Question by Category")

plt.tight_layout()
plt.savefig("tqa_by_category.png", dpi=150, bbox_inches="tight")
plt.show()

# Correlation between category-level avg NE and category accuracy
r_cat, p_cat = stats.pearsonr(cat_stats["avg_ne"], cat_stats["accuracy"])
print(f"Category-level: r(avg_NE, accuracy) = {r_cat:.3f}, p = {p_cat:.4f}")

In [ ]:
# ── CELL 17: Comparison with H1 results ───────────────────────────────────────
# Side-by-side comparison of correlation direction and magnitude
# between Experiment 1 (n=10) and this experiment (n=200+)

H1_RESULTS = {
    "lss":              {"r_h1": +0.64, "confirmed_h1": "4/5"},
    "pronoun_ratio":    {"r_h1": -0.70, "confirmed_h1": "5/5"},
    "ne_count":         {"r_h1": +0.71, "confirmed_h1": "5/5"},
    "perplexity":       {"r_h1": -0.08, "confirmed_h1": "4/5"},
    "semantic_density": {"r_h1": -0.14, "confirmed_h1": "2/5"},
}

print(f"{'Feature':<22} {'H1 r (n=10)':>12} {'H2 r (n='+str(len(df_valid))+')':>14} {'Same direction?':>16} {'H2 significant?'}")
print("-" * 82)

for feat in ["lss", "pronoun_ratio", "ne_count", "perplexity", "semantic_density"]:
    h1 = H1_RESULTS[feat]
    h2_row = next(x for x in corr_results if x["feature"] == feat)
    same_dir = "✅ Yes" if (h1["r_h1"] * h2_row["r"]) > 0 else "❌ Reversed"
    sig      = "✅ Yes" if h2_row["p"] < 0.05 else f"No (p={h2_row['p']:.3f})"
    print(f"{feat:<22} {h1['r_h1']:>12.3f} {h2_row['r']:>14.3f} {same_dir:>16}  {sig}")

In [ ]:
# ── CELL 18: Figure 6 — Full summary dashboard ────────────────────────────────

fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)
fig.suptitle(f"TruthfulQA Scale Experiment — {len(df_valid)} questions — {LLM_MODEL}",
             fontsize=14, fontweight="bold")

plot_feats = [
    ("pronoun_ratio",    "Pronoun Ratio",    False),   # lower = better
    ("ne_count",         "Named Entities",   True),    # higher = better
    ("lss",              "LSS",              True),
    ("perplexity",       "GPT-2 Perplexity", False),
    ("semantic_density", "Semantic Density", True),
    ("token_count",      "Question Length",  None),    # no hypothesis
]

for idx, (feat, title, higher_better) in enumerate(plot_feats):
    ax = fig.add_subplot(gs[idx // 3, idx % 3])
    wrong   = df_valid[df_valid["correct"] == 0][feat]
    correct = df_valid[df_valid["correct"] == 1][feat]

    ax.boxplot([wrong, correct], labels=["Wrong", "Correct"],
               patch_artist=True,
               boxprops={"facecolor": "#e8e8e8"},
               medianprops={"color": "black", "linewidth": 2})

    for vals, color in [(wrong, "#c0392b"), (correct, "#27ae60")]:
        jitter = np.random.normal(0, 0.06, len(vals))
        x_pos  = 1 if color == "#c0392b" else 2
        ax.scatter(x_pos + jitter, vals, color=color, s=10, alpha=0.3, zorder=3)

    r, p = pointbiserialr(df_valid[feat], df_valid["correct"])
    arrow = ("↑ better" if higher_better else "↓ better") if higher_better is not None else ""
    sig   = "*" if p < 0.05 else ""
    ax.set_title(f"{title} {arrow}\nr={r:.3f}, p={p:.3f}{sig}", fontsize=9)

plt.savefig("tqa_summary_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: tqa_summary_dashboard.png")

In [ ]:
# ── CELL 19: Final statistical summary ────────────────────────────────────────

print("=" * 65)
print(" EXPERIMENT 2 — FINAL RESULTS SUMMARY")
print("=" * 65)
print(" Dataset   : TruthfulQA")
print(f" Questions : {len(df_valid)}")
print(f" Model     : {LLM_MODEL}")
print(f" Accuracy  : {df_valid['correct'].mean():.1%}")
print()
print(" Feature correlations with accuracy (point-biserial r):")
print()
for feat in ["pronoun_ratio", "ne_count", "lss", "perplexity",
             "semantic_density", "content_ratio", "token_count"]:
    r, p = pointbiserialr(df_valid[feat], df_valid["correct"])
    sig  = " ← SIGNIFICANT" if p < 0.05 else ""
    print(f"   {feat:<22}  r={r:+.3f}  p={p:.4f}{sig}")
print()
print(" Top 5 logistic regression predictors:")
for _, row in coef_df.head(5).iterrows():
    direction = "higher → more correct" if row["coefficient"] > 0 else "higher → less correct"
    print(f"   {row['feature']:<22}  coef={row['coefficient']:+.3f}  ({direction})")
print("=" * 65)

---
## What to do next (based on results)

### If pronoun_ratio is significant (p < 0.05)
→ Strong replication of H1 finding. Eliminates pronouns from templates is validated at scale.

### If ne_count is significant (p < 0.05)
→ Named entities are confirmed as the primary quality signal. Build NE injection into prompt templates.

### If perplexity is significant (p < 0.05) 
→ GPT-2 surprisal is a valid pre-call quality filter. Can build a cheap prompt screener.

### If semantic_density is NOT significant
→ Confirms H1 finding. Domain-specific embeddings needed. Next experiment: use CodeBERT / BioBERT.

### If nothing is significant
→ TruthfulQA questions may be uniformly specific (it's a benchmark, deliberately structured).
   Next dataset to try: **HotpotQA** (multi-hop reasoning) or **NaturalQuestions** (real Google queries).

---
*Research notebook — mycontext/research/hypothesis_2_truthfulqa_scale.ipynb*